In [ ]:
import json
import time
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer
)
import evaluate
from sentence_transformers import SentenceTransformer, util
import re
import string
from collections import Counter


# ==========================================
# 1. Load and Prepare Data
# ==========================================

json_file = "version1_2_fixed_other.json"   # Change to your file

with open(json_file, "r", encoding="utf-8") as f:
    data = json.load(f)["data"]

flat_data = []
sample_id = 0

for item in data:
    title = item.get("title", "")

    for para in item.get("paragraphs", []):
        context = para.get("context", "")

        for qa in para.get("qas", []):

            # Skip questions without answers
            if not qa.get("answers"):
                continue

            answer = qa["answers"][0]

            flat_data.append({
                "id": qa.get("id", f"sample_{sample_id}"),
                "title": title,
                "context": context,
                "question": qa.get("question", ""),
                "answer_text": answer.get("text", ""),
                "answer_start": answer.get("answer_start", -1),
                "answer_end": answer.get("answer_end", -1)
            })

            sample_id += 1

from sklearn.model_selection import train_test_split

train_raw, test_raw = train_test_split(flat_data, test_size=0.2, random_state=42)

train_dataset = Dataset.from_list(train_raw)
test_dataset = Dataset.from_list(test_raw)

print(f"✅ Loaded {len(flat_data[:2000])} total samples")
print(f"✅ Training samples: {len(train_dataset)}")
print(f"✅ Testing samples: {len(test_dataset)}")
print(train_dataset[0])

# ==========================================
# 2. Tokenization and Preprocessing
# ==========================================

# CHANGED: Use Qwen2-0.5B (unsloth build) instead of RoBERTa
model_checkpoint = "unsloth/Qwen2-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# CHANGED: Qwen2's tokenizer has no pad token by default (it's a decoder-only
# BPE tokenizer), so we fall back to eos_token for padding.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def preprocess_function(examples):
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=384,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    tokenized["start_positions"] = []
    tokenized["end_positions"] = []
    tokenized["answer_texts"] = []
    tokenized["example_id"] = []  # NEW: rebuilt per output row

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]

        # CHANGED: Qwen2 has no CLS/BOS-style token to anchor "no answer" cases
        # on (unlike RoBERTa's <s>/BERT's [CLS]). We use index 0 as the
        # fallback "no answer" position instead.
        no_answer_index = 0

        sample_index = sample_mapping[i]
        answer_text = examples["answer_text"][sample_index]
        answer_start_char = examples["answer_start"][sample_index]
        answer_end_char = examples["answer_end"][sample_index]

        tokenized["answer_texts"].append(answer_text)
        tokenized["example_id"].append(examples["id"][sample_index])

        sequence_ids = tokenized.sequence_ids(i)

        # Find the context part (sequence_ids == 1)
        try:
            context_start = sequence_ids.index(1)
            # Find last occurrence of 1
            context_end = len(sequence_ids) - 1 - sequence_ids[::-1].index(1)
        except ValueError:
            # If no context tokens found (rare edge case), default to no_answer_index
            tokenized["start_positions"].append(no_answer_index)
            tokenized["end_positions"].append(no_answer_index)
            continue

        # Check if the answer is entirely within the context span of this feature
        if offsets[context_start][0] > answer_end_char or offsets[context_end][1] < answer_start_char:
            tokenized["start_positions"].append(no_answer_index)
            tokenized["end_positions"].append(no_answer_index)
        else:
            token_start = context_start
            while token_start <= context_end and offsets[token_start][0] <= answer_start_char:
                token_start += 1
            start_position = token_start - 1

            token_end = context_end
            while token_end >= context_start and offsets[token_end][1] >= answer_end_char:
                token_end -= 1
            end_position = token_end + 1

            tokenized["start_positions"].append(start_position)
            tokenized["end_positions"].append(end_position)

    return tokenized


# Tokenize train and test SEPARATELY to prevent stride leakage
cols_to_remove_train = [col for col in train_dataset.column_names if col != "answer_texts"]
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=cols_to_remove_train)

cols_to_remove_test = [col for col in test_dataset.column_names if col != "answer_texts"]
tokenized_test = test_dataset.map(preprocess_function, batched=True, remove_columns=cols_to_remove_test)

print(f"✅ Tokenized train chunks: {len(tokenized_train)}")
print(f"✅ Tokenized test chunks: {len(tokenized_test)}")
# ==========================================
# 3. Model Initialization & Training
# ==========================================
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

# CHANGED: Qwen2 was initialized without a pad token id set on the model config;
# align it with the tokenizer so batched training doesn't error out.
if model.config.pad_token_id is None:
    model.config.pad_token_id = tokenizer.pad_token_id

start_train_time = time.time()

training_args = TrainingArguments(
    output_dir="./results_qwen2",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1,
    weight_decay=0.01,
    report_to="none",
    use_cpu=True,
    logging_steps=1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,   # ✅ Train on the training chunks
    eval_dataset=tokenized_test,     # ✅ Validate on unseen test chunks

)

print("Starting Training...")
trainer.train()

end_train_time = time.time()
training_time = end_train_time - start_train_time
print(f"Training completed in {training_time:.2f} seconds.")

# ==========================================
# 3.5 Plot Training Loss
# ==========================================
log_history = trainer.state.log_history
steps = []
losses = []

for log in log_history:
    if "loss" in log:
        steps.append(log["step"])
        losses.append(log["loss"])

if steps:
    plt.figure(figsize=(10, 5))
    plt.plot(steps, losses, linewidth=2)
    plt.title("Training Loss over Steps (Qwen2-0.5B)")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    plt.savefig("training_loss_qwen2.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
else:
    print("No loss history found to plot.")

# ==========================================
# 4. Comprehensive Evaluation Metrics (String-Based QA)
# ==========================================
print("Evaluating Model...")


def normalize_text(s):
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))


def compute_qa_f1(prediction, ground_truth):
    prediction_tokens = normalize_text(prediction).split()
    ground_truth_tokens = normalize_text(ground_truth).split()
    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = 1.0 * num_same / len(prediction_tokens)
    recall = 1.0 * num_same / len(ground_truth_tokens)
    return 2 * (precision * recall) / (precision + recall)


def compute_qa_exact_match(prediction, ground_truth):
    return 1.0 if normalize_text(prediction) == normalize_text(ground_truth) else 0.0


bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")

similarity_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')


def evaluate_model(model, dataset, tokenizer):
    device = torch.device("cpu")
    model.to(device)
    model.eval()

    all_preds_text = []
    all_refs_text = []
    confidences = []
    similarities = []

    with torch.no_grad():
        for i in range(len(dataset)):
            sample = dataset[i]

            # CHANGED: Qwen2 does NOT use token_type_ids either, same as RoBERTa.
            valid_keys = ['input_ids', 'attention_mask']

            inputs = {
                k: torch.tensor(v).unsqueeze(0).to(device)
                for k, v in sample.items() if k in valid_keys
            }

            outputs = model(**inputs)
            start_logits = outputs.start_logits
            end_logits = outputs.end_logits

            start_idx = torch.argmax(start_logits, dim=1).item()
            end_idx = torch.argmax(end_logits, dim=1).item()

            if end_idx < start_idx:
                end_idx = start_idx

            start_probs = torch.softmax(start_logits, dim=1)
            end_probs = torch.softmax(end_logits, dim=1)
            confidence = (start_probs[0, start_idx] * end_probs[0, end_idx]).item()
            confidences.append(confidence)

            pred_tokens = inputs['input_ids'][0, start_idx:end_idx + 1]
            pred_text = tokenizer.decode(pred_tokens, skip_special_tokens=True).strip()

            ref_text = sample['answer_texts']

            all_preds_text.append(pred_text)
            all_refs_text.append(ref_text)

            # Handle empty predictions for similarity model
            if not pred_text:
                sim_score = 0.0
            else:
                pred_emb = similarity_model.encode(pred_text, convert_to_tensor=True, device='cpu')
                ref_emb = similarity_model.encode(ref_text, convert_to_tensor=True, device='cpu')
                sim_score = util.cos_sim(pred_emb, ref_emb).item()

            similarities.append(sim_score)

    f1_scores = [compute_qa_f1(pred, ref) for pred, ref in zip(all_preds_text, all_refs_text)]
    em_scores = [compute_qa_exact_match(pred, ref) for pred, ref in zip(all_preds_text, all_refs_text)]

    avg_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0.0
    avg_em = sum(em_scores) / len(em_scores) if em_scores else 0.0

    bleu_score = bleu_metric.compute(predictions=all_preds_text, references=[[r] for r in all_refs_text])['bleu']
    rouge_score = rouge_metric.compute(predictions=all_preds_text, references=all_refs_text, rouge_types=["rougeL"])['rougeL']
    meteor_score = meteor_metric.compute(predictions=all_preds_text, references=all_refs_text)['meteor']

    avg_confidence = float(np.mean(confidences)) if confidences else 0.0
    avg_similarity = float(np.mean(similarities)) if similarities else 0.0

    return {
        "F1 Score (Token Overlap)": avg_f1,
        "Accuracy (Exact Match)": avg_em,
        "BLEU": bleu_score,
        "ROUGE-L": rouge_score,
        "METEOR": meteor_score,
        "Avg Confidence": avg_confidence,
        "Avg Semantic Similarity": avg_similarity,
        "Training Time (seconds)": training_time
    }


# ✅ Final evaluation ONLY on the unseen test set
results = evaluate_model(model, tokenized_test, tokenizer)
print("\n" + "=" * 45)
print("FINAL EVALUATION RESULTS (Qwen2-0.5B)")
print("=" * 45)
for metric, value in results.items():
    if isinstance(value, float):
        print(f"{metric:<25}: {value:.4f}")
    else:
        print(f"{metric:<25}: {value}")

In [ ]:
import json
import time
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer
)
import evaluate
from sentence_transformers import SentenceTransformer, util
from peft import LoraConfig, get_peft_model, TaskType
import re
import string
from collections import Counter


# ==========================================
# 1. Load and Prepare Data
# ==========================================

json_file = "version1_2_fixed_other.json"   # Change to your file

with open(json_file, "r", encoding="utf-8") as f:
    data = json.load(f)["data"]

flat_data = []
sample_id = 0

for item in data:
    title = item.get("title", "")

    for para in item.get("paragraphs", []):
        context = para.get("context", "")

        for qa in para.get("qas", []):

            # Skip questions without answers
            if not qa.get("answers"):
                continue

            answer = qa["answers"][0]

            flat_data.append({
                "id": qa.get("id", f"sample_{sample_id}"),
                "title": title,
                "context": context,
                "question": qa.get("question", ""),
                "answer_text": answer.get("text", ""),
                "answer_start": answer.get("answer_start", -1),
                "answer_end": answer.get("answer_end", -1)
            })

            sample_id += 1

from sklearn.model_selection import train_test_split

train_raw, test_raw = train_test_split(flat_data, test_size=0.2, random_state=42)

train_dataset = Dataset.from_list(train_raw)
test_dataset = Dataset.from_list(test_raw)

print(f"✅ Loaded {len(flat_data[:2000])} total samples")
print(f"✅ Training samples: {len(train_dataset)}")
print(f"✅ Testing samples: {len(test_dataset)}")
print(train_dataset[0])
# ==========================================
# 2. Tokenization and Preprocessing
# ==========================================

# CHANGED: Use Qwen2-0.5B (unsloth build) instead of RoBERTa
model_checkpoint = "unsloth/Qwen2-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# CHANGED: Qwen2's tokenizer has no pad token by default (it's a decoder-only
# BPE tokenizer), so we fall back to eos_token for padding.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def preprocess_function(examples):
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=384,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    tokenized["start_positions"] = []
    tokenized["end_positions"] = []
    tokenized["answer_texts"] = []
    tokenized["example_id"] = []  # NEW: rebuilt per output row

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]

        # CHANGED: Qwen2 has no CLS/BOS-style token to anchor "no answer" cases
        # on (unlike RoBERTa's <s>/BERT's [CLS]). We use index 0 as the
        # fallback "no answer" position instead.
        no_answer_index = 0

        sample_index = sample_mapping[i]
        answer_text = examples["answer_text"][sample_index]
        answer_start_char = examples["answer_start"][sample_index]
        answer_end_char = examples["answer_end"][sample_index]

        tokenized["answer_texts"].append(answer_text)
        tokenized["example_id"].append(examples["id"][sample_index])

        sequence_ids = tokenized.sequence_ids(i)

        # Find the context part (sequence_ids == 1)
        try:
            context_start = sequence_ids.index(1)
            # Find last occurrence of 1
            context_end = len(sequence_ids) - 1 - sequence_ids[::-1].index(1)
        except ValueError:
            # If no context tokens found (rare edge case), default to no_answer_index
            tokenized["start_positions"].append(no_answer_index)
            tokenized["end_positions"].append(no_answer_index)
            continue

        # Check if the answer is entirely within the context span of this feature
        if offsets[context_start][0] > answer_end_char or offsets[context_end][1] < answer_start_char:
            tokenized["start_positions"].append(no_answer_index)
            tokenized["end_positions"].append(no_answer_index)
        else:
            token_start = context_start
            while token_start <= context_end and offsets[token_start][0] <= answer_start_char:
                token_start += 1
            start_position = token_start - 1

            token_end = context_end
            while token_end >= context_start and offsets[token_end][1] >= answer_end_char:
                token_end -= 1
            end_position = token_end + 1

            tokenized["start_positions"].append(start_position)
            tokenized["end_positions"].append(end_position)

    return tokenized


# FIX: remove ALL original columns
# Tokenize train and test SEPARATELY to prevent stride leakage
cols_to_remove_train = [col for col in train_dataset.column_names if col != "answer_texts"]
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=cols_to_remove_train)

cols_to_remove_test = [col for col in test_dataset.column_names if col != "answer_texts"]
tokenized_test = test_dataset.map(preprocess_function, batched=True, remove_columns=cols_to_remove_test)

print(f"✅ Tokenized train chunks: {len(tokenized_train)}")
print(f"✅ Tokenized test chunks: {len(tokenized_test)}")
# ==========================================
# 3. Model Initialization & Training
# ==========================================
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

# CHANGED: Qwen2 was initialized without a pad token id set on the model config;
# align it with the tokenizer so batched training doesn't error out.
if model.config.pad_token_id is None:
    model.config.pad_token_id = tokenizer.pad_token_id

# ==========================================
# 3.1 Wrap Model with LoRA (PEFT)
# ==========================================
# NOTE: task_type=QUESTION_ANS tells PEFT to also unfreeze/train the
# QA output head (qa_outputs), since that layer is randomly initialized
# and LoRA adapters alone won't learn a from-scratch head.
lora_config = LoraConfig(
    task_type=TaskType.QUESTION_ANS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Qwen2 attention projections
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

start_train_time = time.time()

training_args = TrainingArguments(
    output_dir="./results_qwen2_lora",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    report_to="none",
    use_cpu=True,
    logging_steps=1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,   # ✅ Train on the training chunks
    eval_dataset=tokenized_test,     # ✅ Validate on unseen test chunks
)

print("Starting Training...")
trainer.train()

end_train_time = time.time()
training_time = end_train_time - start_train_time
print(f"Training completed in {training_time:.2f} seconds.")

# ==========================================
# 3.5 Plot Training Loss
# ==========================================
log_history = trainer.state.log_history
steps = []
losses = []

for log in log_history:
    if "loss" in log:
        steps.append(log["step"])
        losses.append(log["loss"])

if steps:
    plt.figure(figsize=(10, 5))
    plt.plot(steps, losses, linewidth=2)
    plt.title("Training Loss over Steps (Qwen2-0.5B)")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    plt.savefig("training_loss_qwen2lora.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
else:
    print("No loss history found to plot.")

# ==========================================
# 4. Comprehensive Evaluation Metrics (String-Based QA)
# ==========================================
print("Evaluating Model...")


def normalize_text(s):
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))


def compute_qa_f1(prediction, ground_truth):
    prediction_tokens = normalize_text(prediction).split()
    ground_truth_tokens = normalize_text(ground_truth).split()
    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = 1.0 * num_same / len(prediction_tokens)
    recall = 1.0 * num_same / len(ground_truth_tokens)
    return 2 * (precision * recall) / (precision + recall)


def compute_qa_exact_match(prediction, ground_truth):
    return 1.0 if normalize_text(prediction) == normalize_text(ground_truth) else 0.0


bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")

similarity_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')


def evaluate_model(model, dataset, tokenizer):
    device = torch.device("cpu")
    model.to(device)
    model.eval()

    all_preds_text = []
    all_refs_text = []
    confidences = []
    similarities = []

    with torch.no_grad():
        for i in range(len(dataset)):
            sample = dataset[i]

            # CHANGED: Qwen2 does NOT use token_type_ids either, same as RoBERTa.
            valid_keys = ['input_ids', 'attention_mask']

            inputs = {
                k: torch.tensor(v).unsqueeze(0).to(device)
                for k, v in sample.items() if k in valid_keys
            }

            outputs = model(**inputs)
            start_logits = outputs.start_logits
            end_logits = outputs.end_logits

            start_idx = torch.argmax(start_logits, dim=1).item()
            end_idx = torch.argmax(end_logits, dim=1).item()

            if end_idx < start_idx:
                end_idx = start_idx

            start_probs = torch.softmax(start_logits, dim=1)
            end_probs = torch.softmax(end_logits, dim=1)
            confidence = (start_probs[0, start_idx] * end_probs[0, end_idx]).item()
            confidences.append(confidence)

            pred_tokens = inputs['input_ids'][0, start_idx:end_idx + 1]
            pred_text = tokenizer.decode(pred_tokens, skip_special_tokens=True).strip()

            ref_text = sample['answer_texts']

            all_preds_text.append(pred_text)
            all_refs_text.append(ref_text)

            # Handle empty predictions for similarity model
            if not pred_text:
                sim_score = 0.0
            else:
                pred_emb = similarity_model.encode(pred_text, convert_to_tensor=True, device='cpu')
                ref_emb = similarity_model.encode(ref_text, convert_to_tensor=True, device='cpu')
                sim_score = util.cos_sim(pred_emb, ref_emb).item()

            similarities.append(sim_score)

    f1_scores = [compute_qa_f1(pred, ref) for pred, ref in zip(all_preds_text, all_refs_text)]
    em_scores = [compute_qa_exact_match(pred, ref) for pred, ref in zip(all_preds_text, all_refs_text)]

    avg_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0.0
    avg_em = sum(em_scores) / len(em_scores) if em_scores else 0.0

    bleu_score = bleu_metric.compute(predictions=all_preds_text, references=[[r] for r in all_refs_text])['bleu']
    rouge_score = rouge_metric.compute(predictions=all_preds_text, references=all_refs_text, rouge_types=["rougeL"])['rougeL']
    meteor_score = meteor_metric.compute(predictions=all_preds_text, references=all_refs_text)['meteor']

    avg_confidence = float(np.mean(confidences)) if confidences else 0.0
    avg_similarity = float(np.mean(similarities)) if similarities else 0.0

    return {
        "F1 Score (Token Overlap)": avg_f1,
        "Accuracy (Exact Match)": avg_em,
        "BLEU": bleu_score,
        "ROUGE-L": rouge_score,
        "METEOR": meteor_score,
        "Avg Confidence": avg_confidence,
        "Avg Semantic Similarity": avg_similarity,
        "Training Time (seconds)": training_time
    }


# ✅ Final evaluation ONLY on the unseen test set
results = evaluate_model(model, tokenized_test, tokenizer)
print("\n" + "=" * 45)
print("FINAL EVALUATION RESULTS (Qwen2-0.5B)")
print("=" * 45)
for metric, value in results.items():
    if isinstance(value, float):
        print(f"{metric:<25}: {value:.4f}")
    else:
        print(f"{metric:<25}: {value}")

In [ ]:
import json
import time
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    TrainerCallback
)
import evaluate
from sentence_transformers import SentenceTransformer, util
from peft import AdaLoraConfig, get_peft_model, TaskType
import re
import string
from collections import Counter


# ==========================================
# 1. Load and Prepare Data
# ==========================================

json_file = "dataset_QA.json"   # Change to your file

with open(json_file, "r", encoding="utf-8") as f:
    data = json.load(f)["data"]

flat_data = []
sample_id = 0

for item in data:
    title = item.get("title", "")

    for para in item.get("paragraphs", []):
        context = para.get("context", "")

        for qa in para.get("qas", []):

            # Skip questions without answers
            if not qa.get("answers"):
                continue

            answer = qa["answers"][0]

            flat_data.append({
                "id": qa.get("id", f"sample_{sample_id}"),
                "title": title,
                "context": context,
                "question": qa.get("question", ""),
                "answer_text": answer.get("text", ""),
                "answer_start": answer.get("answer_start", -1),
                "answer_end": answer.get("answer_end", -1)
            })

            sample_id += 1

from sklearn.model_selection import train_test_split

# Split the data: 80% train, 20% test
train_raw, test_raw = train_test_split(flat_data, test_size=0.2, random_state=42)

train_dataset = Dataset.from_list(train_raw)
test_dataset = Dataset.from_list(test_raw)

print(f"✅ Loaded {len(flat_data)} total samples")
print(f"✅ Training samples: {len(train_dataset)}")
print(f"✅ Testing samples: {len(test_dataset)}")
print(train_dataset[0])
# ==========================================
# 2. Tokenization and Preprocessing
# ==========================================

# CHANGED: Use Qwen2-0.5B (unsloth build) instead of RoBERTa
model_checkpoint = "unsloth/Qwen2-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# CHANGED: Qwen2's tokenizer has no pad token by default (it's a decoder-only
# BPE tokenizer), so we fall back to eos_token for padding.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def preprocess_function(examples):
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=384,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    tokenized["start_positions"] = []
    tokenized["end_positions"] = []
    tokenized["answer_texts"] = []
    tokenized["example_id"] = []  # NEW: rebuilt per output row

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]

        # CHANGED: Qwen2 has no CLS/BOS-style token to anchor "no answer" cases
        # on (unlike RoBERTa's <s>/BERT's [CLS]). We use index 0 as the
        # fallback "no answer" position instead.
        no_answer_index = 0

        sample_index = sample_mapping[i]
        answer_text = examples["answer_text"][sample_index]
        answer_start_char = examples["answer_start"][sample_index]
        answer_end_char = examples["answer_end"][sample_index]

        tokenized["answer_texts"].append(answer_text)
        tokenized["example_id"].append(examples["id"][sample_index])

        sequence_ids = tokenized.sequence_ids(i)

        # Find the context part (sequence_ids == 1)
        try:
            context_start = sequence_ids.index(1)
            # Find last occurrence of 1
            context_end = len(sequence_ids) - 1 - sequence_ids[::-1].index(1)
        except ValueError:
            # If no context tokens found (rare edge case), default to no_answer_index
            tokenized["start_positions"].append(no_answer_index)
            tokenized["end_positions"].append(no_answer_index)
            continue

        # Check if the answer is entirely within the context span of this feature
        if offsets[context_start][0] > answer_end_char or offsets[context_end][1] < answer_start_char:
            tokenized["start_positions"].append(no_answer_index)
            tokenized["end_positions"].append(no_answer_index)
        else:
            token_start = context_start
            while token_start <= context_end and offsets[token_start][0] <= answer_start_char:
                token_start += 1
            start_position = token_start - 1

            token_end = context_end
            while token_end >= context_start and offsets[token_end][1] >= answer_end_char:
                token_end -= 1
            end_position = token_end + 1

            tokenized["start_positions"].append(start_position)
            tokenized["end_positions"].append(end_position)

    return tokenized


# Tokenize train and test SEPARATELY to prevent stride leakage
cols_to_remove_train = [col for col in train_dataset.column_names if col != "answer_texts"]
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=cols_to_remove_train)

cols_to_remove_test = [col for col in test_dataset.column_names if col != "answer_texts"]
tokenized_test = test_dataset.map(preprocess_function, batched=True, remove_columns=cols_to_remove_test)

print(f"✅ Tokenized train chunks: {len(tokenized_train)}")
print(f"✅ Tokenized test chunks: {len(tokenized_test)}")
# ==========================================
# 3. Model Initialization & Training
# ==========================================
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

# CHANGED: Qwen2 was initialized without a pad token id set on the model config;
# align it with the tokenizer so batched training doesn't error out.
if model.config.pad_token_id is None:
    model.config.pad_token_id = tokenizer.pad_token_id

# ==========================================
# 3.1 Wrap Model with AdaLoRA (PEFT)
# ==========================================
# AdaLoRA needs to know the total number of optimizer steps up front. Training
# is split into three phases: `tinit` warmup steps at full initial rank, then
# a pruning phase where rank is reduced down to target_r, then `tfinal` steps
# at the fixed final budget. NOTE: tinit and tfinal are both *durations*
# (number of steps), not absolute step numbers -- the pruning phase runs from
# step `tinit` to step `total_step - tfinal`, so tinit + tfinal must leave
# room for at least one deltaT-sized pruning update in between.
per_device_batch_size = 2
num_train_epochs = 3
# ✅ Use ONLY the training chunks to calculate AdaLoRA total steps
steps_per_epoch = max(1, len(tokenized_train) // per_device_batch_size)
total_steps = max(1, steps_per_epoch * num_train_epochs)
if total_steps < 30:
    # Too few steps for a meaningful three-phase schedule (e.g. a tiny
    # dataset/smoke test) -- skip straight to pruning from step 0.
    tinit, tfinal, deltaT = 0, 0, 1
else:
    tinit = total_steps // 10
    tfinal = total_steps // 10
    deltaT = 10

assert tinit < total_steps - tfinal, (
    f"AdaLoRA schedule invalid: tinit={tinit}, tfinal={tfinal}, total_steps={total_steps}. "
    "Increase total_steps (more data/epochs) or lower tinit/tfinal."
)

# NOTE: task_type=QUESTION_ANS tells PEFT to also unfreeze/train the
# QA output head (qa_outputs), since that layer is randomly initialized
# and adapters alone won't learn a from-scratch head.
adalora_config = AdaLoraConfig(
    task_type=TaskType.QUESTION_ANS,

    # ✅ FIX 1: Increase rank budget significantly for 0.5B scale
    init_r=24,          # Was 12. Need more headroom for pruning to find optimal allocation
    target_r=16,        # Was 8. Minimum viable rank for Qwen2-0.5B ERP adaptation

    # ✅ FIX 2: Match alpha to target_r (ratio ≤ 2.0)
    lora_alpha=32,      # 32/16 = 2.0 scaling factor (safe upper bound)
    lora_dropout=0.05,  # Keep as-is, appropriate for small model

    # ✅ FIX 3: Expand target modules for generative QA
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"   # Critical for AdaLoRA redistribution
    ],

    # ✅ FIX 4: Calibrate pruning schedule to actual training length
    tinit=int(total_steps * 0.1),     # Warmup 10% of training before pruning begins
    tfinal=int(total_steps * 0.1),    # Hold final budget for last 10% for stabilization
    deltaT=max(1, total_steps // 20), # Update rank mask ~20 times during training

    beta1=0.85,
    beta2=0.85,
    total_step=total_steps,
)

model = get_peft_model(model, adalora_config)
model.print_trainable_parameters()


# AdaLoRA requires update_and_allocate(global_step) to be called every training
# step, using the gradients from that step, to update the rank budget/mask.
# It must run on `on_optimizer_step` (fires right after optimizer.step(), but
# BEFORE gradients are zeroed) -- NOT `on_step_end`, which fires after
# gradients have already been zeroed and would leave p.grad as None.
class AdaLoraUpdateCallback(TrainerCallback):
    def on_optimizer_step(self, args, state, control, **kwargs):
        model.base_model.update_and_allocate(state.global_step)


start_train_time = time.time()

training_args = TrainingArguments(
    output_dir="./results_qwen2_adalora",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=per_device_batch_size,
    per_device_eval_batch_size=per_device_batch_size,
    num_train_epochs=1,
    weight_decay=0.01,
    report_to="none",
    use_cpu=True,
    logging_steps=1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    callbacks=[AdaLoraUpdateCallback()],
)

print("Starting Training...")
trainer.train()

end_train_time = time.time()
training_time = end_train_time - start_train_time
print(f"Training completed in {training_time:.2f} seconds.")

# ==========================================
# 3.5 Plot Training Loss
# ==========================================
log_history = trainer.state.log_history
steps = []
losses = []

for log in log_history:
    if "loss" in log:
        steps.append(log["step"])
        losses.append(log["loss"])

if steps:
    plt.figure(figsize=(10, 5))
    plt.plot(steps, losses, linewidth=2)
    plt.title("Training Loss over Steps (Qwen2-0.5B)")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    plt.savefig("training_loss_qwen2adalora.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
else:
    print("No loss history found to plot.")

# ==========================================
# 4. Comprehensive Evaluation Metrics (String-Based QA)
# ==========================================
print("Evaluating Model...")


def normalize_text(s):
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))


def compute_qa_f1(prediction, ground_truth):
    prediction_tokens = normalize_text(prediction).split()
    ground_truth_tokens = normalize_text(ground_truth).split()
    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = 1.0 * num_same / len(prediction_tokens)
    recall = 1.0 * num_same / len(ground_truth_tokens)
    return 2 * (precision * recall) / (precision + recall)


def compute_qa_exact_match(prediction, ground_truth):
    return 1.0 if normalize_text(prediction) == normalize_text(ground_truth) else 0.0


bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")

similarity_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')


def evaluate_model(model, dataset, tokenizer):
    device = torch.device("cpu")
    model.to(device)
    model.eval()

    all_preds_text = []
    all_refs_text = []
    confidences = []
    similarities = []

    with torch.no_grad():
        for i in range(len(dataset)):
            sample = dataset[i]

            # CHANGED: Qwen2 does NOT use token_type_ids either, same as RoBERTa.
            valid_keys = ['input_ids', 'attention_mask']

            inputs = {
                k: torch.tensor(v).unsqueeze(0).to(device)
                for k, v in sample.items() if k in valid_keys
            }

            outputs = model(**inputs)
            start_logits = outputs.start_logits
            end_logits = outputs.end_logits

            start_idx = torch.argmax(start_logits, dim=1).item()
            end_idx = torch.argmax(end_logits, dim=1).item()

            if end_idx < start_idx:
                end_idx = start_idx

            start_probs = torch.softmax(start_logits, dim=1)
            end_probs = torch.softmax(end_logits, dim=1)
            confidence = (start_probs[0, start_idx] * end_probs[0, end_idx]).item()
            confidences.append(confidence)

            pred_tokens = inputs['input_ids'][0, start_idx:end_idx + 1]
            pred_text = tokenizer.decode(pred_tokens, skip_special_tokens=True).strip()

            ref_text = sample['answer_texts']

            all_preds_text.append(pred_text)
            all_refs_text.append(ref_text)

            # Handle empty predictions for similarity model
            if not pred_text:
                sim_score = 0.0
            else:
                pred_emb = similarity_model.encode(pred_text, convert_to_tensor=True, device='cpu')
                ref_emb = similarity_model.encode(ref_text, convert_to_tensor=True, device='cpu')
                sim_score = util.cos_sim(pred_emb, ref_emb).item()

            similarities.append(sim_score)

    f1_scores = [compute_qa_f1(pred, ref) for pred, ref in zip(all_preds_text, all_refs_text)]
    em_scores = [compute_qa_exact_match(pred, ref) for pred, ref in zip(all_preds_text, all_refs_text)]

    avg_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0.0
    avg_em = sum(em_scores) / len(em_scores) if em_scores else 0.0

    bleu_score = bleu_metric.compute(predictions=all_preds_text, references=[[r] for r in all_refs_text])['bleu']
    rouge_score = rouge_metric.compute(predictions=all_preds_text, references=all_refs_text, rouge_types=["rougeL"])['rougeL']
    meteor_score = meteor_metric.compute(predictions=all_preds_text, references=all_refs_text)['meteor']

    avg_confidence = float(np.mean(confidences)) if confidences else 0.0
    avg_similarity = float(np.mean(similarities)) if similarities else 0.0

    return {
        "F1 Score (Token Overlap)": avg_f1,
        "Accuracy (Exact Match)": avg_em,
        "BLEU": bleu_score,
        "ROUGE-L": rouge_score,
        "METEOR": meteor_score,
        "Avg Confidence": avg_confidence,
        "Avg Semantic Similarity": avg_similarity,
        "Training Time (seconds)": training_time
    }


results = evaluate_model(model, tokenized_test, tokenizer)

print("\n" + "=" * 45)
print("FINAL EVALUATION RESULTS (Qwen2-0.5B + AdaLoRA)")
print("=" * 45)
for metric, value in results.items():
    if isinstance(value, float):
        print(f"{metric:<25}: {value:.4f}")
    else:
        print(f"{metric:<25}: {value}")

In [ ]:
pip install evaluate